# 🚚 DataCo Smart Supply Chain — Late Delivery Risk (Leakage-Aware) with CatBoost
What this notebook is
A practical, end-to-end machine learning notebook using the DataCo Smart Supply Chain dataset to predict Late_delivery_risk (binary classification: late vs on-time).

The emphasis is not just on “getting a high score,” but on building a model that is valid at prediction time:

we actively audit and remove leakage,

we remove PII / near-unique identifiers that encourage memorization,

we choose an operating threshold based on the business trade-off between false alarms and missed late deliveries,

we compare a simple baseline to a stronger tabular model.

What we did (so far)
1) Loaded and inspected the dataset
Read DataCoSupplyChainDataset.csv and examined columns that could serve as targets.

Identified likely target candidates and selected a clean binary target:

Target: Late_delivery_risk (values: 0/1)

2) Built a leakage-aware feature set
Started with a conservative rule: remove columns that directly encode outcomes or are only known after fulfillment.

Discovered “too good to be true” performance in an early tree model run, which prompted a deeper leakage/PII audit.

3) Baseline model: Logistic Regression + preprocessing pipeline
Built a scikit-learn pipeline that handles mixed types:

numeric: median impute + scaling

categorical: most-frequent impute + one-hot encoding

Achieved a solid baseline:

ROC-AUC ≈ 0.848

4) Threshold tuning (turning probabilities into decisions)
Evaluated multiple thresholds and selected a threshold that improves recall for late deliveries:

Chosen threshold: 0.40

At threshold 0.40 (Logistic Regression): higher recall, but many false alarms.

5) Strong model: CatBoost (gradient-boosted decision trees)
Chosen because it performs well on tabular data and can use categorical features effectively.

Fixed CatBoost categorical requirements:

categorical columns must be string/int and not NaN → filled missing with "__MISSING__".

6) Removed PII / identifiers / “status-like” leakage
To make the model realistic and avoid memorization, we dropped several columns including:

Customer Fname, Customer Lname, Customer Street, Customer Email, Customer Password

Customer Id, Order Customer Id, Product Card Id

Order Status, Product Status

Product Name, Product Image

This reduced the feature set to a cleaner, more deployable subset.

7) Results after cleaning
With the cleaned feature set (no obvious PII/IDs/status leakage):

Clean CatBoost ROC-AUC ≈ 0.9866

At threshold = 0.40 (clean CatBoost):

FP = 1,641 (false alarms)

FN = 944 (missed late shipments)

Precision (late) ≈ 0.920

Recall (late) ≈ 0.952

8) Mini-experiment: tree depth sweep
We tested CatBoost depth as a controlled experiment:

depth 4 → AUC ≈ 0.9785

depth 6 → AUC ≈ 0.9824

depth 8 → AUC ≈ 0.9850

depth 10 → AUC ≈ 0.9866

Takeaway: deeper trees captured more feature interactions in this dataset, but depth 7–8 is a practical “industry sweet spot” balancing complexity and performance.

Why this notebook is useful practice
This notebook explicitly exercises core ML engineering judgment:

detecting and removing leakage (when performance is suspiciously high),

deciding what features are acceptable at prediction time,

choosing thresholds based on operational trade-offs,

comparing a simple baseline with a strong production-grade tabular model.

Next steps (optional)
Feature importance review for remaining leakage signals (e.g., profit/sales fields)

Group/time-aware validation (avoid near-duplicate leakage)

Error analysis: inspect top false negatives and false positives

Probability calibration if probabilities will be used for prioritization



# Step 1 — Load + quick schema/quality scan 

In [1]:
import pandas as pd

base_path = "/kaggle/input/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis"

data_path = f"{base_path}/DataCoSupplyChainDataset.csv"
desc_path = f"{base_path}/DescriptionDataCoSupplyChain.csv"
logs_path = f"{base_path}/tokenized_access_logs.csv"

data = pd.read_csv(data_path, encoding="latin1", low_memory=False)
description = pd.read_csv(desc_path, encoding="latin1", low_memory=False)
access_logs = pd.read_csv(logs_path, low_memory=False)

print("data shape:", data.shape)
print("description shape:", description.shape)
print("access_logs shape:", access_logs.shape)

display(data.head(3))
display(description.head(10))
display(access_logs.head(3))

# Quick column overview
overview = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_rate": data.isna().mean(),
    "n_unique": data.nunique(dropna=True),
}).sort_values(["missing_rate", "n_unique"], ascending=[False, False])

display(overview.head(30))

# Show columns that look like candidate labels/targets
candidate_targets = [c for c in data.columns if any(
    key in c.lower() for key in ["late", "delay", "delivery", "shipping", "status", "days", "real", "scheduled"]
)]
print("Candidate target-ish columns:")
for c in candidate_targets:
    print(" -", c)


data shape: (180519, 53)
description shape: (52, 2)
access_logs shape: (469977, 8)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


,FIELDS,DESCRIPTION
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased...
3,Benefit per order,: Earnings per order placed
4,Sales per customer,: Total sales per customer made per customer
5,Delivery Status,: Delivery status of orders: Advance shipping...
6,Late_delivery_risk,: Categorical variable that indicates if send...
7,Category Id,: Product category code
8,Category Name,: Description of the product category
9,Customer City,: City where the customer made the purchase


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...


,dtype,missing_rate,n_unique
Product Description,float64,1.000000,0
Order Zipcode,float64,0.862397,609
Customer Lname,object,0.000044,1109
Customer Zipcode,float64,0.000017,995
Order Item Id,int64,0.000000,180519
order date (DateOrders),object,0.000000,65752
Order Id,int64,0.000000,65752
shipping date (DateOrders),object,0.000000,63701
Benefit per order,float64,0.000000,21998
Order Profit Per Order,float64,0.000000,21998


Candidate target-ish columns:
 - Days for shipping (real)
 - Days for shipment (scheduled)
 - Delivery Status
 - Late_delivery_risk
 - Order Status
 - Product Status
 - shipping date (DateOrders)
 - Shipping Mode


# Step 2 — Define label + remove obvious leakage columns

In [2]:
label_column = "Late_delivery_risk"

# Columns that typically leak the answer (derived after shipping/delivery happens)
leakage_columns = [
    "Delivery Status",                 # usually reflects delivered/late etc.
    "Days for shipping (real)",        # uses actual shipping outcome
    "shipping date (DateOrders)",      # can encode timeline; we’ll revisit later
]

# Keep scheduled days (known at order time), drop real days (known after)
# Also drop the label from features
feature_columns = [c for c in data.columns if c not in leakage_columns + [label_column]]

X = data[feature_columns].copy()
y = data[label_column].copy()

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts(dropna=False))


X shape: (180519, 49)
y value counts:
 Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


# Tell me what values the label uses

In [3]:
print("Label dtype:", y.dtype)
print("Unique label values (sample):", y.dropna().unique()[:20])


Label dtype: int64
Unique label values (sample): [0 1]


In [4]:
# Drop columns that are 100% missing
all_null_columns = [c for c in X.columns if X[c].isna().all()]
print("Dropping all-null columns:", all_null_columns)

X = X.drop(columns=all_null_columns)


Dropping all-null columns: ['Product Description']


# Step 3 — Logistic Regression baseline + metrics

In [5]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Column types
numeric_columns = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [c for c in X_train.columns if c not in numeric_columns]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_columns),
    ],
    remainder="drop"
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, n_jobs=-1)),
])

pipeline.fit(X_train, y_train)

y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


ROC-AUC: 0.8479414389044437
Confusion matrix:
 [[12688  3620]
 [ 5022 14774]]
              precision    recall  f1-score   support

           0      0.716     0.778     0.746     16308
           1      0.803     0.746     0.774     19796

    accuracy                          0.761     36104
   macro avg      0.760     0.762     0.760     36104
weighted avg      0.764     0.761     0.761     36104



# Step 4 — Threshold tuning (precision vs recall trade-off)

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.linspace(0.1, 0.9, 17)

rows = []
for threshold in thresholds:
    y_pred_t = (y_proba >= threshold).astype(int)
    rows.append({
        "threshold": float(threshold),
        "precision": precision_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
        "f1": f1_score(y_test, y_pred_t),
        "positive_rate": y_pred_t.mean(),
    })

results = pd.DataFrame(rows).sort_values("f1", ascending=False)
display(results.head(10))


,threshold,precision,recall,f1,positive_rate
6,0.40,0.737247,0.832289,0.781891,0.618990
5,0.35,0.706317,0.874369,0.781409,0.678761
7,0.45,0.771461,0.788998,0.780131,0.560769
4,0.30,0.674976,0.910992,0.775422,0.740029
8,0.50,0.803197,0.746312,0.773710,0.509473
3,0.25,0.647326,0.941756,0.767265,0.797696
9,0.55,0.832497,0.704233,0.763012,0.463827
2,0.20,0.621904,0.966660,0.756872,0.852260
10,0.60,0.859304,0.659931,0.746536,0.421089
1,0.15,0.600105,0.983431,0.745372,0.898543


In [7]:
best = results.iloc[0]
best_threshold = best["threshold"]
print("Best-F1 threshold:", best_threshold)
print(best)


Best-F1 threshold: 0.4
threshold        0.400000
precision        0.737247
recall           0.832289
f1               0.781891
positive_rate    0.618990
Name: 6, dtype: float64


# Step 5 — Evaluate at the chosen threshold (0.40)

In [8]:
from sklearn.metrics import confusion_matrix, classification_report

chosen_threshold = 0.40

y_pred_040 = (y_proba >= chosen_threshold).astype(int)

print("Threshold:", chosen_threshold)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_040))
print(classification_report(y_test, y_pred_040, digits=3))


Threshold: 0.4
Confusion matrix:
 [[10436  5872]
 [ 3320 16476]]
              precision    recall  f1-score   support

           0      0.759     0.640     0.694     16308
           1      0.737     0.832     0.782     19796

    accuracy                          0.745     36104
   macro avg      0.748     0.736     0.738     36104
weighted avg      0.747     0.745     0.742     36104



# Step 6 — Quick “business framing” numbers

In [9]:
cm = confusion_matrix(y_test, y_pred_040)
tn, fp, fn, tp = cm.ravel()

print("TP (caught late):", tp)
print("FN (missed late):", fn)
print("FP (false alarms):", fp)
print("TN (correct on-time):", tn)

print("Recall (late caught):", tp / (tp + fn))
print("Precision (alerts correct):", tp / (tp + fp))
print("Alert rate:", (tp + fp) / (tn + fp + fn + tp))


TP (caught late): 16476
FN (missed late): 3320
FP (false alarms): 5872
TN (correct on-time): 10436
Recall (late caught): 0.832289351384118
Precision (alerts correct): 0.7372471809557902
Alert rate: 0.6189895856414802


# Step 7 — Upgrade to a boosted tree model

In [10]:
import importlib

for pkg in ["xgboost", "lightgbm", "catboost"]:
    spec = importlib.util.find_spec(pkg)
    print(pkg, "OK" if spec is not None else "NOT INSTALLED")


xgboost OK
lightgbm OK
catboost OK


# Step 8 — CatBoost model (handles categoricals) + ROC-AUC

In [11]:
from sklearn.model_selection import train_test_split

X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = [
    column for column in X_train_cb.columns
    if X_train_cb[column].dtype == "object" or str(X_train_cb[column].dtype).startswith("category")
]

# Fill missing values in categoricals, and cast to string to be safe
for column in categorical_features:
    X_train_cb[column] = X_train_cb[column].fillna("__MISSING__").astype(str)
    X_test_cb[column] = X_test_cb[column].fillna("__MISSING__").astype(str)

print("Categorical columns fixed:", len(categorical_features))


Categorical columns fixed: 22


In [12]:
from sklearn.model_selection import train_test_split
from catboost import Pool, CatBoostClassifier

# Split
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Recompute categoricals safely: treat non-numeric columns as categorical
categorical_columns_cb = X_train_cb.select_dtypes(exclude=["number"]).columns.tolist()

# Convert categoricals to string and fill missing
X_train_cb = X_train_cb.copy()
X_test_cb = X_test_cb.copy()

for column in categorical_columns_cb:
    X_train_cb[column] = X_train_cb[column].fillna("__MISSING__").astype(str)
    X_test_cb[column] = X_test_cb[column].fillna("__MISSING__").astype(str)

# Pass categorical feature indices (CatBoost likes indices)
categorical_feature_indices = [X_train_cb.columns.get_loc(column) for column in categorical_columns_cb]

print("Categorical columns:", len(categorical_columns_cb))
print("Example categorical columns:", categorical_columns_cb[:10])


Categorical columns: 22
Example categorical columns: ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State']


In [13]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

train_pool = Pool(X_train_cb, y_train_cb, cat_features=categorical_feature_indices)
test_pool = Pool(X_test_cb, y_test_cb, cat_features=categorical_feature_indices)

cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model.fit(train_pool, eval_set=test_pool, use_best_model=True)

y_proba_cb = cat_model.predict_proba(X_test_cb)[:, 1]
print("CatBoost ROC-AUC:", roc_auc_score(y_test_cb, y_proba_cb))

chosen_threshold = 0.40
y_pred_cb = (y_proba_cb >= chosen_threshold).astype(int)

print("Confusion matrix @0.40:\n", confusion_matrix(y_test_cb, y_pred_cb))
print(classification_report(y_test_cb, y_pred_cb, digits=3))


0:	test: 0.9948438	best: 0.9948438 (0)	total: 584ms	remaining: 7m 46s
100:	test: 0.9954988	best: 0.9955316 (36)	total: 45.4s	remaining: 5m 14s
200:	test: 0.9955233	best: 0.9955316 (36)	total: 1m 33s	remaining: 4m 37s
300:	test: 0.9955318	best: 0.9955534 (280)	total: 2m 21s	remaining: 3m 54s
400:	test: 0.9955511	best: 0.9955563 (396)	total: 3m 11s	remaining: 3m 10s
500:	test: 0.9955704	best: 0.9956010 (454)	total: 4m	remaining: 2m 23s
600:	test: 0.9955500	best: 0.9956010 (454)	total: 4m 50s	remaining: 1m 36s
700:	test: 0.9956074	best: 0.9956074 (700)	total: 5m 41s	remaining: 48.2s
799:	test: 0.9955973	best: 0.9956169 (723)	total: 6m 31s	remaining: 0us

bestTest = 0.9956169002
bestIteration = 723

Shrink model to first 724 iterations.
CatBoost ROC-AUC: 0.9956169001817061
Confusion matrix @0.40:
 [[15253  1055]
 [  618 19178]]
              precision    recall  f1-score   support

           0      0.961     0.935     0.948     16308
           1      0.948     0.969     0.958     19796



# find leakage by looking at top features

In [14]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat_model.get_feature_importance(train_pool)
}).sort_values("importance", ascending=False)

display(importance.head(30))


,feature,importance
25,order date (DateOrders),35.799582
47,Shipping Mode,17.802841
39,Order Status,5.374423
22,Order City,4.397002
9,Customer Fname,3.792065
21,Market,2.948485
4,Category Id,1.965698
17,Department Id,1.835288
0,Type,1.707301
23,Order Country,1.493451


# Step 9 — Drop PII/IDs/status + convert order date to safe time features

In [15]:
import pandas as pd

Xclean = X.copy()

dropcols = [
    "Customer Fname", "Customer Lname", "Customer Street", "Customer Password", "Customer Email",
    "Customer Id", "Order Customer Id", "Product Card Id",
    "Order Status", "Product Status",
    "Product Name", "Product Image",
]

dropcols = [c for c in dropcols if c in Xclean.columns]
print("Dropping:", dropcols)

Xclean = Xclean.drop(columns=dropcols)

if "order date (DateOrders)" in Xclean.columns:
    dt = pd.to_datetime(Xclean["order date (DateOrders)"], errors="coerce")
    Xclean["orderDayOfWeek"] = dt.dt.dayofweek
    Xclean["orderMonth"] = dt.dt.month
    Xclean["orderHour"] = dt.dt.hour
    Xclean = Xclean.drop(columns=["order date (DateOrders)"])

print("Xclean shape:", Xclean.shape)


Dropping: ['Customer Fname', 'Customer Lname', 'Customer Street', 'Customer Password', 'Customer Email', 'Customer Id', 'Order Customer Id', 'Product Card Id', 'Order Status', 'Product Status', 'Product Name', 'Product Image']
Xclean shape: (180519, 38)


In [16]:
from sklearn.model_selection import train_test_split
from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

Xtrain, Xtest, ytrain, ytest = train_test_split(
    Xclean, y, test_size=0.2, random_state=42, stratify=y
)

catcols = Xtrain.select_dtypes(exclude=["number"]).columns.tolist()

Xtrain = Xtrain.copy()
Xtest = Xtest.copy()
for col in catcols:
    Xtrain[col] = Xtrain[col].fillna("__MISSING__").astype(str)
    Xtest[col] = Xtest[col].fillna("__MISSING__").astype(str)

catidx = [Xtrain.columns.get_loc(col) for col in catcols]

trainPool = Pool(Xtrain, ytrain, cat_features=catidx)
testPool = Pool(Xtest, ytest, cat_features=catidx)

model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)

model.fit(trainPool, eval_set=testPool, use_best_model=True)


proba = model.predict_proba(Xtest)[:, 1]
print("Clean CatBoost ROC-AUC:", roc_auc_score(ytest, proba))

threshold = 0.40
pred = (proba >= threshold).astype(int)
print("Confusion matrix @0.40:\n", confusion_matrix(ytest, pred))
print(classification_report(ytest, pred, digits=3))


0:	test: 0.7738104	best: 0.7738104 (0)	total: 236ms	remaining: 4m 42s
200:	test: 0.9795737	best: 0.9795769 (199)	total: 58s	remaining: 4m 48s
400:	test: 0.9840939	best: 0.9840976 (390)	total: 2m 1s	remaining: 4m 2s
600:	test: 0.9849439	best: 0.9849516 (593)	total: 3m 5s	remaining: 3m 4s
800:	test: 0.9855350	best: 0.9855350 (800)	total: 4m 10s	remaining: 2m 4s
1000:	test: 0.9861439	best: 0.9861447 (999)	total: 5m 15s	remaining: 1m 2s
1199:	test: 0.9866471	best: 0.9866475 (1198)	total: 6m 20s	remaining: 0us

bestTest = 0.9866474686
bestIteration = 1198

Shrink model to first 1199 iterations.
Clean CatBoost ROC-AUC: 0.9866474686392818
Confusion matrix @0.40:
 [[14667  1641]
 [  944 18852]]
              precision    recall  f1-score   support

           0      0.940     0.899     0.919     16308
           1      0.920     0.952     0.936     19796

    accuracy                          0.928     36104
   macro avg      0.930     0.926     0.927     36104
weighted avg      0.929     0.92

# Note: micro-experiment with "depth" in catboost

In [17]:
for d in [4, 6, 8, 10]:
    model = CatBoostClassifier(
        iterations=600,
        learning_rate=0.05,
        depth=d,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False
    )
    model.fit(trainPool, eval_set=testPool, use_best_model=True)
    proba = model.predict_proba(Xtest)[:, 1]
    print("depth", d, "AUC", roc_auc_score(ytest, proba))


depth 4 AUC 0.9785413839509823
depth 6 AUC 0.9824258020477004
depth 8 AUC 0.9849515586329097
depth 10 AUC 0.9865557525365548
